<!-- journey-badges -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NolaNexus/LMp2LMa/blob/main/track-04-ml-architect/08_data_architecture.ipynb)
[![Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/NolaNexus/LMp2LMa/blob/main/track-04-ml-architect/08_data_architecture.ipynb)
[![Open in GitHub Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/NolaNexus/LMp2LMa)

# 08 — Data Architecture (Medallion Layers)

**Objectives**
- Model bronze → silver → gold layers with explicit contracts.
- Make ingestion idempotent (dedupe by event key).
- Validate schema changes before they reach downstream consumers.


In [ ]:
import pandas as pd

bronze = pd.DataFrame(
    {
        "event_id": ["e1", "e2", "e2", "e3", "e4"],
        "user_id": ["u1", "u1", "u1", "u2", "u2"],
        "event": ["view", "click", "click", "purchase", "view"],
        "amount": [0.0, 0.0, 0.0, 42.5, 0.0],
        "ingested_at": pd.date_range("2026-01-01", periods=5, freq="min"),
    }
)
print("bronze_rows:", len(bronze))
print(bronze.head())

In [ ]:
# Silver: enforce the data contract, then deduplicate on event_id
REQUIRED_COLUMNS = {"event_id", "user_id", "event", "amount"}


def validate(df: pd.DataFrame) -> list[str]:
    problems = []
    missing = REQUIRED_COLUMNS - set(df.columns)
    if missing:
        problems.append(f"missing columns: {sorted(missing)}")
    if (df["amount"] < 0).any():
        problems.append("negative amounts")
    if df["event_id"].isna().any():
        problems.append("null event ids")
    return problems


problems = validate(bronze)
print("contract_violations:", problems)

silver = bronze.drop_duplicates(subset="event_id", keep="last").copy()
print("silver_rows:", len(silver))

In [ ]:
# Gold: business aggregates with a stable grain
gold = silver.groupby("user_id", as_index=False).agg(
    events=("event_id", "count"), revenue=("amount", "sum")
)
print(gold)
print("gold_grain: one row per user")

In [ ]:
# Schema evolution check: additive changes are safe, removals are breaking
old_schema = {"event_id", "user_id", "event", "amount"}
new_schema = {"event_id", "user_id", "event", "amount", "device"}

print("breaking_change:", bool(old_schema - new_schema))
print("additive_fields:", sorted(new_schema - old_schema))

**Design notes**
- Bronze is append-only raw truth; never mutate it.
- Silver owns contracts and dedupe; replaying bronze must reproduce silver exactly.
- Gold is disposable: rebuild from silver whenever definitions change.
